# TP6 — Sac de mots, TF-IDF et mini moteur de recherche
EMSI — TALN · Solution complète

## Partie 0 — Préparation
### 0.1 Identification

In [28]:
NUM_TP = 6
# 🪪 IDENTIFICATION — à vérifier / adapter avant de rendre
NOM = "MOUSSAID"
PRENOM = "Mehdi"
CLASSE = "5IIR"      # ← à adapter
GROUPE = "G2"        # ← à adapter
SITE = "Roudani"      # ← à adapter : "Roudani", "Maarif", "Centre", "CFC", "Oranger", "Moulay Youssef"
SITES_EMSI = ["Roudani", "Maarif", "Centre", "CFC", "Oranger", "Moulay Youssef"]
assert NOM and PRENOM and CLASSE and GROUPE, "Complétez NOM, PRENOM, CLASSE et GROUPE."
assert SITE in SITES_EMSI, f"SITE doit être l'un de {SITES_EMSI}"
NOM_FICHIER = f"TP{NUM_TP}_{NOM.upper().replace(' ', '-')}_{PRENOM.capitalize()}_{CLASSE}_{GROUPE}_{SITE.replace(' ', '')}.ipynb"
print(f"Étudiant(e) : {PRENOM} {NOM.upper()} | Classe : {CLASSE} | Groupe : {GROUPE} | EMSI {SITE}")
print("➡️ Nom du fichier à rendre :", NOM_FICHIER)

Étudiant(e) : Mehdi MOUSSAID | Classe : 5IIR | Groupe : G2 | EMSI Roudani
➡️ Nom du fichier à rendre : TP6_MOUSSAID_Mehdi_5IIR_G2_Roudani.ipynb


### 0.2 Installation et imports

In [29]:
# Installation (une seule fois)
%pip install -q scikit-learn nltk pandas numpy

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.1.2 -> 26.3
[notice] To update, run: C:\Users\admin\scoop\apps\python\current\python.exe -m pip install --upgrade pip


In [30]:
import os
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"  # évite le plantage du noyau (conflit OpenMP numpy / scikit-learn sous Windows-Anaconda)
import warnings
warnings.filterwarnings("ignore")
import re, html, math
from collections import Counter
import numpy as np
import pandas as pd
import nltk
nltk.download("stopwords", quiet=True)
from nltk.corpus import stopwords
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
pd.set_option("display.max_colwidth", 90)
def verifier(description, test):
    """Affiche ✅ si test() est vrai, ❌ sinon (sans interrompre le notebook)."""
    try:
        ok = bool(test())
    except Exception as e:
        print(f"❌ {description} — {type(e).__name__} : {e}")
        return
    print(("✅ " if ok else "❌ ") + description)
print("Imports réussis")

Imports réussis


### 0.3 Le corpus et la chaîne de prétraitement (fournis)

In [31]:
AVIS = [
("Le tajine était délicieux et le service très rapide. Je recommande !", "pos"),
("Service TRÈS lent, le plat est arrivé froid. Je ne recommande pas.", "neg"),
("Les serveurs sont aimables et le cadre est magnifique 😍", "pos"),
("Le couscous n'était pas bon du tout, trop salé.", "neg"),
("Trooop bien !!! Livraison rapide et repas encore chaud.", "pos"),
("Livraison en retard de 45 minutes, la commande était incomplète.", "neg"),
("Très bon rapport qualité-prix, les portions sont généreuses.", "pos"),
("Le restaurant est sale et le personnel n'est pas aimable.", "neg"),
("J'ai adoré le thé à la menthe et les pâtisseries marocaines.", "pos"),
("Les prix sont trop élevés pour des plats sans goût.", "neg"),
("Accueil chaleureux, on s'est régalés en famille. Merci @dar_tajine !", "pos"),
("Plus jamais ! Le poisson n'était pas frais.", "neg"),
("Cadre agréable, musique douce, serveurs souriants.", "pos"),
("Commande annulée sans explication, service client injoignable.", "neg"),
("Le meilleur restaurant de Rabat, les brochettes sont excellentes.", "pos"),
("Les frites étaient froides et la sauce n'avait aucun goût.", "neg"),
("tres bon resto, je reviendrai avec mes amis", "pos"),
("Pas bon, pas chaud, pas rapide : rien ne va.", "neg"),
("Réservation facile sur https://dar-tajine.ma et accueil parfait.", "pos"),
("Le service était lent mais les plats étaient bons.", "pos"),
("Je ne suis pas satisfait, le livreur était impoli.", "neg"),
("Une expérience inoubliable, tout était parfait !", "pos"),
("Le restaurent est magnifque, les serveurs très gentils.", "pos"),
("Les plats sont arrivés froids, je ne commanderai plus jamais.", "neg"),
("Le plat est bon.", "pos"),
("Le plat n'est pas bon.", "neg"),
]
AVIS_COURS = [
("Service rapide et personnel souriant, je recommande ce restaurant.", "pos"),
("Le tajine au poulet était délicieux, service rapide.", "pos"),
("Très bon rapport qualité prix, le couscous était parfait.", "pos"),
("Service lent, le plat était froid. Je ne recommande pas.", "neg"),
("Le tajine au poulet est excellent et pas cher.", "pos"),
("Très bon accueil, service rapide, je recommande.", "pos"),
("Le couscous était froid et le service lent.", "neg"),
("Rapport qualité prix correct, mais le service est lent.", "neg"),
("Personnel souriant, tajine au poulet délicieux, très bon moment.", "pos"),
("Pas cher, très bon, je recommande ce restaurant à Rabat.", "pos"),
]
AVIS_NOUVEAUX = [
("Le tajine au poulet et aux olives était parfait, service rapide et souriant.", "pos"),
("Attente interminable, service lent et serveur désagréable. Je ne recommande pas.", "neg"),
("Très bon couscous royal, la viande était tendre et bien cuite.", "pos"),
("La pastilla au poulet était froide et beaucoup trop sucrée.", "neg"),
("Excellent rapport qualité prix, je recommande ce restaurant les yeux fermés.", "pos"),
("Le thé à la menthe était délicieux, accueil chaleureux comme à la maison.", "pos"),
("Livraison rapide mais la pizza était froide et sans goût.", "neg"),
("Personnel souriant, cadre agréable et terrasse magnifique avec vue sur la mer.", "pos"),
("Prix trop élevés pour des portions minuscules, plus jamais !", "neg"),
("Le service était rapide et les plats bien présentés, très bon moment en famille.", "pos"),
("Nous avons attendu une heure, la commande était fausse et personne ne s'est excusé.", "neg"),
("Les brochettes de kefta étaient excellentes et pas cher du tout.", "pos"),
("Le poisson n'était pas frais et les frites étaient trop grasses.", "neg"),
("Très bon accueil, service rapide, le tajine de kefta était délicieux.", "pos"),
("Restaurant sale, toilettes dégoûtantes, je ne reviendrai jamais.", "neg"),
("La harira était excellente, servie bien chaude avec des dattes.", "pos"),
("Le service est lent et le serveur a oublié la moitié de la commande.", "neg"),
("Cadre magnifique et musique andalouse, une très belle soirée à Fès.", "pos"),
("Les plats étaient froids et le pain était sec. Très déçu.", "neg"),
("Très bon rapport qualité prix, les salades marocaines sont fraîches et variées.", "pos"),
("Livraison en retard, le livreur était impoli et le repas froid.", "neg"),
("Le meilleur couscous de Casablanca, je recommande vivement !", "pos"),
("Pas cher mais pas bon : la viande était dure et trop salée.", "neg"),
("Accueil chaleureux, personnel souriant et plats délicieux, bravo à toute l'équipe.", "pos"),
("Le service était correct mais les plats n'avaient aucun goût.", "neg"),
("Jus d'orange frais et crêpes marocaines délicieuses au petit déjeuner.", "pos"),
("Commande annulée deux fois, service client injoignable, plus jamais !", "neg"),
("Le tajine au poulet était délicieux et le service très rapide.", "pos"),
("Musique trop forte, service lent et addition trop salée.", "neg"),
("Les serveurs sont très aimables et le cadre est très agréable.", "pos"),
("La viande n'était pas cuite et le serveur ne nous a même pas écoutés.", "neg"),
("Très bon restaurant, les desserts sont délicieux et pas cher.", "pos"),
("Attente trop longue pour une table, puis service lent. Dommage.", "neg"),
("Une pastilla au poisson délicieuse et un thé à la menthe parfait.", "pos"),
("Les frites étaient froides et le burger sans goût, je ne recommande pas.", "neg"),
("Terrasse agréable, service rapide et prix raisonnables à Rabat.", "pos"),
("Le couscous était froid et la sauce trop salée, très déçu.", "neg"),
("Personnel souriant et efficace, tout était parfait, merci !", "pos"),
("Hygiène douteuse, la cuisine était sale et les verres mal lavés.", "neg"),
("tres bon tajine, service rapide, je recommande 👍", "pos"),
("Le serveur était désagréable et la commande est arrivée en retard.", "neg"),
("Très bon moment entre amis, les grillades étaient délicieuses.", "pos"),
("Rapport qualité prix décevant, les portions sont trop petites.", "neg"),
("Accueil chaleureux et service rapide, le tajine au poulet est un régal.", "pos"),
("Le plat est arrivé froid et le service était lent, quelle déception.", "neg"),
("Livraison rapide et repas encore chaud, je recommande ce restaurant.", "pos"),
("Pas de climatisation, salle bruyante et service lent.", "neg"),
("La pastilla au poulet est la meilleure de Marrakech, très bon accueil.", "pos"),
("Les prix ont augmenté mais la qualité a baissé, plus jamais.", "neg"),
("Délicieux couscous aux sept légumes, pas cher et copieux.", "pos"),
("Le thé était froid et les pâtisseries pas fraîches, déçu.", "neg"),
("Service impeccable, plats délicieux et cadre magnifique, bravo !", "pos"),
("Réservation perdue, attente d'une heure, je ne recommande pas ce restaurant.", "neg"),
("Excellent accueil, le chef est venu nous saluer, très bon souvenir.", "pos"),
("Le tajine de poisson manquait de sel et les légumes étaient trop cuits.", "neg"),
("Rapport qualité prix imbattable, très bon restaurant familial.", "pos"),
("Service lent, plats froids et serveurs débordés. Dommage !!!", "neg"),
("Jus frais, salades variées et service rapide, très bon petit restaurant.", "pos"),
("La commande était incomplète et la livraison en retard d'une heure.", "neg"),
("Les brochettes étaient délicieuses et le personnel très souriant.", "pos"),
("Je ne recommande pas : viande dure, frites froides, prix élevés.", "neg"),
("Ambiance familiale, accueil chaleureux et tajine au poulet délicieux.", "pos"),
("Le service était rapide mais le couscous n'avait aucun goût.", "neg"),
("Très bon rapport qualité prix et service rapide, je recommande !", "pos"),
]
CORPUS = AVIS + AVIS_COURS + AVIS_NOUVEAUX  # le corpus fil rouge complet
textes = [texte for texte, _ in CORPUS]
etiquettes = [etiquette for _, etiquette in CORPUS]
print(len(AVIS), "avis du TP3 +", len(AVIS_COURS), "avis du cours +", len(AVIS_NOUVEAUX), "nouveaux avis =", len(CORPUS), "avis")
print(Counter(etiquettes))
# ---- chaîne du TP2 (corrigé) ----
def espaces(t):
    return re.sub(r"\s+", " ", t).strip()
def retirer_html(t):
    return espaces(re.sub(r"<[^>]+>", " ", html.unescape(t)))
JETONS = [
    (r"https?://\S+|www\.\S+", " <URL> "),
    (r"[\w.+-]+@[\w-]+(?:\.[\w-]+)+", " <EMAIL> "),
    (r"(?<!\w)@\w+", " <USER> "),
    (r"#(\w+)", r" \1 "),
    (r"(?<!\d)(?:0|\+212 ?)[67](?:[ -]?\d{2}){4}(?!\d)", " <TEL> "),
]
def remplacer_jetons(t):
    for motif, remplacement in JETONS:
        t = re.sub(motif, remplacement, t)
    return espaces(t)
def reduire_repetitions(t):
    t = re.sub(r"([^\W\d_])\1{2,}", r"\1\1", t)
    return re.sub(r"([!?.])\1+", r"\1", t)
EMOJI = r"[\U0001F300-\U0001FAFF\u2600-\u27BF]"
EMOJIS_POS = {"😍", "👍", "😋", "🎉", "🎂"}
EMOJIS_NEG = {"😡", "👎", "😞", "😒", "☹"}
def jeton_emoji(m):
    e = m.group()
    return " <POS> " if e in EMOJIS_POS else " <NEG> " if e in EMOJIS_NEG else " <EMOJI> "
def traiter_emojis(t, mode="jetons"):
    if mode == "garder":
        return t
    if mode == "retirer":
        return espaces(re.sub(EMOJI, " ", t))
    return espaces(re.sub(EMOJI, jeton_emoji, t))
def nettoyer(t, emojis="jetons"):
    t = retirer_html(t)
    t = remplacer_jetons(t)
    t = reduire_repetitions(t)
    return espaces(traiter_emojis(t, emojis))
JETON = r"<[A-Z]+>"
PARTIES = [r"[Aa]ujourd['’]hui", JETON, r"(?:[A-Z]\.){2,}", r"(?:M|Mme|Dr|etc)\.",
           r"\d+(?:[.,]\d+)?(?![^\W\d_])", r"[^\W\d_]+['’]", r"\w+(?:-\w+)*", r"[^\w\s]"]
TOKEN = re.compile("|".join(PARTIES))
def tok_regex(t):
    return TOKEN.findall(t)
def preprocess(texte, emojis="jetons", minuscules=True):
    """Texte brut → liste de tokens propres (livrable du TP2)."""
    tokens = tok_regex(nettoyer(texte, emojis))
    if minuscules:
        tokens = [t if re.fullmatch(JETON, t) else t.lower() for t in tokens]
    return tokens
NEGATION = {"ne", "n", "pas", "plus", "jamais", "rien", "aucun", "sans", "ni"}
MOTS_VIDES = (set(stopwords.words("french")) - NEGATION) | {"restaurant", "resto", "plat", "plats"}
def analyseur(texte):
    """preprocess() du TP2, sans ponctuation ni mots vides adaptés (TP3)."""
    return [t for t in preprocess(texte)
            if re.search(r"\w", t) and t.rstrip("'’") not in MOTS_VIDES]
print(textes[1]); print(" ->", analyseur(textes[1]))
print(textes[18]); print(" ->", analyseur(textes[18]))

26 avis du TP3 + 10 avis du cours + 64 nouveaux avis = 100 avis
Counter({'pos': 54, 'neg': 46})
Service TRÈS lent, le plat est arrivé froid. Je ne recommande pas.
 -> ['service', 'très', 'lent', 'arrivé', 'froid', 'ne', 'recommande', 'pas']
Réservation facile sur https://dar-tajine.ma et accueil parfait.
 -> ['réservation', 'facile', '<URL>', 'accueil', 'parfait']


## Partie 1 — Le sac de mots
### Exercice 1.1 ★ CountVectorizer par défaut

In [32]:
vec_brut = CountVectorizer()
X_brut = vec_brut.fit_transform(textes)
densite = X_brut.nnz / np.prod(X_brut.shape)
print("matrice :", X_brut.shape, f"| densité : {densite:.1%}")
print(vec_brut.get_feature_names_out()[:15])

matrice : (100, 252) | densité : 3.8%
['45' 'accueil' 'addition' 'adoré' 'agréable' 'ai' 'aimable' 'aimables'
 'ambiance' 'amis' 'andalouse' 'annulée' 'arrivé' 'arrivée' 'arrivés']


In [33]:
# ✅ Vérification (ne pas modifier)
verifier("100 lignes", lambda: X_brut.shape[0] == 100)
verifier("densité", lambda: abs(densite - X_brut.nnz / np.prod(X_brut.shape)) < 1e-12)

✅ 100 lignes
✅ densité


### Exercice 1.2 ★★ Avec notre prétraitement

In [34]:
vec_bow = CountVectorizer(analyzer=analyseur)
X_bow = vec_bow.fit_transform(textes)
mots_bow = vec_bow.get_feature_names_out()
totaux = np.asarray(X_bow.sum(axis=0)).ravel()
print("vocabulaire : par défaut", X_brut.shape[1], "mots | avec analyseur", X_bow.shape[1], "mots")
print([(mots_bow[j], int(totaux[j])) for j in np.argsort(totaux)[::-1][:10]])

vocabulaire : par défaut 252 mots | avec analyseur 218 mots
[('service', 30), ('très', 26), ('pas', 26), ('bon', 23), ('rapide', 18), ('recommande', 15), ('ne', 12), ('trop', 12), ('lent', 12), ('prix', 12)]


In [35]:
# ✅ Vérification (ne pas modifier)
verifier("X_bow utilise l'analyseur", lambda: X_bow.shape == (100, len(set(t for x in textes for t in analyseur(x)))))
verifier("totaux", lambda: len(totaux) == X_bow.shape[1] and totaux.sum() == X_bow.sum())

✅ X_bow utilise l'analyseur
✅ totaux


In [36]:
# Aide pour l'interprétation 1(b) : différences entre les deux vocabulaires
sb, sa = set(vec_brut.get_feature_names_out()), set(mots_bow)
print("Gardés par défaut, retirés par l'analyseur :", sorted(sb - sa))
print("\nGardés par l'analyseur, absents du défaut :", sorted(sa - sb))
print(f"\nDensité avec l'analyseur : {X_bow.nnz / np.prod(X_bow.shape):.1%}")

Gardés par défaut, retirés par l'analyseur : ['ai', 'au', 'aux', 'avaient', 'avait', 'avec', 'avons', 'ce', 'dar', 'dar_tajine', 'de', 'des', 'du', 'en', 'est', 'et', 'https', 'je', 'la', 'le', 'les', 'ma', 'mais', 'mes', 'même', 'nous', 'on', 'ont', 'plat', 'plats', 'pour', 'restaurant', 'resto', 'sont', 'suis', 'sur', 'trooop', 'un', 'une', 'étaient', 'était']

Gardés par l'analyseur, absents du défaut : ['<POS>', '<URL>', '<USER>', 'a', "n'", 'qualité-prix', 'troop']

Densité avec l'analyseur : 2.9%


### 📝 Interprétation 1
**(a)** La densité de la matrice par défaut est de **3,8 %** (252 colonnes pour 100 avis). Une matrice « creuse » est une matrice dont presque toutes les cases valent 0 : un avis ne contient qu'une dizaine de mots différents sur les 252 du vocabulaire, donc plus de 96 % des cases sont nulles. Avec notre analyseur la densité tombe à ≈ 2,9 % (218 mots). On stocke donc la matrice au format *sparse* (seules les cases non nulles sont gardées), ce qui économise mémoire et temps de calcul, surtout avec un vrai vocabulaire de dizaines de milliers de mots.

**(b)** Le `CountVectorizer` par défaut garde les **mots vides** (*le, la, les, et, est, de, des, du, je, nous, sont, était, étaient, avait, mais, pour, avec…*), les **mots du domaine** que l'on avait retirés (*plat, plats, restaurant, resto*), et il casse les URL et les pseudos en morceaux sans signification (*https*, *dar*, *dar_tajine*). Il garde aussi le mot déformé *trooop* tel quel.
À l'inverse, l'analyseur garde ce que le défaut perd : le **« n' »** de la négation (le motif par défaut ignore les jetons d'un seul caractère), les **jetons spéciaux** `<URL>`, `<USER>` et `<POS>` (l'emoji 😍 est transformé en `<POS>`, le défaut l'ignore), **« qualité-prix »** en un seul mot, **« troop »** (répétitions de lettres réduites) et le mot « a » (verbe). Résultat : un vocabulaire plus petit (218 contre 252), débarrassé du bruit, mais qui conserve la négation et les informations normalisées (URL, mention, emoji).

## Partie 2 — TF-IDF
### Exercice 2.1 ★★ Vérifier le calcul à la main du cours

In [37]:
docs_cours = [["tajine", "délicieux", "service", "rapide"],
              ["tajine", "froid", "service", "lent"],
              ["service", "rapide", "service", "souriant"]]
def idf_cours(docs_tok):
    N = len(docs_tok)
    vocab = {m for d in docs_tok for m in d}
    return {m: math.log10(N / sum(m in d for d in docs_tok)) for m in vocab}
idf = idf_cours(docs_cours)
print({m: round(v, 3) for m, v in sorted(idf.items())})

{'délicieux': 0.477, 'froid': 0.477, 'lent': 0.477, 'rapide': 0.176, 'service': 0.0, 'souriant': 0.477, 'tajine': 0.176}


In [38]:
# ✅ Vérification (ne pas modifier)
verifier("service : 0", lambda: abs(idf["service"]) < 1e-12)
verifier("rapide : 0,176 ; souriant : 0,477", lambda: round(idf["rapide"], 3) == 0.176 and round(idf["souriant"], 3) == 0.477)

✅ service : 0
✅ rapide : 0,176 ; souriant : 0,477


### Exercice 2.2 ★ TfidfVectorizer sur le corpus

In [39]:
vec_tfidf = TfidfVectorizer(analyzer=analyseur)
X_tfidf = vec_tfidf.fit_transform(textes)
mots = vec_tfidf.get_feature_names_out()
idf_sk = dict(zip(mots, vec_tfidf.idf_))
for m in ["service", "rapide", "pas", "délicieux", "kefta", "climatisation"]:
    print(f"{m:14} df = {int(np.sum(X_tfidf[:, vec_tfidf.vocabulary_[m]].toarray() > 0)):2} idf = {idf_sk[m]:.2f}")

service        df = 30 idf = 2.18
rapide         df = 18 idf = 2.67
pas            df = 22 idf = 2.48
délicieux      df = 11 idf = 3.13
kefta          df =  2 idf = 4.52
climatisation  df =  1 idf = 4.92


In [40]:
# ✅ Vérification (ne pas modifier)
verifier("X_tfidf", lambda: X_tfidf.shape == X_bow.shape)
verifier("lignes normalisées (longueur 1)", lambda: np.allclose(np.linalg.norm(X_tfidf.toarray(), axis=1), 1))

✅ X_tfidf
✅ lignes normalisées (longueur 1)


### Exercice 2.3 ★★ Les mots caractéristiques d'un avis

In [41]:
def mots_caracteristiques(i, k=3):
    ligne = X_tfidf[i].toarray()[0]
    meilleurs = np.argsort(ligne)[::-1][:k]
    return [(mots[j], round(float(ligne[j]), 2)) for j in meilleurs]
for i in (0, 3, 40, 63):
    print(i, textes[i]); print("  ->", mots_caracteristiques(i))

0 Le tajine était délicieux et le service très rapide. Je recommande !
  -> [('tajine', 0.47), ('délicieux', 0.47), ('recommande', 0.42)]
3 Le couscous n'était pas bon du tout, trop salé.
  -> [('salé', 0.53), ('tout', 0.43), ('couscous', 0.37)]
40 Excellent rapport qualité prix, je recommande ce restaurant les yeux fermés.
  -> [('yeux', 0.48), ('fermés', 0.48), ('excellent', 0.41)]
63 Le tajine au poulet était délicieux et le service très rapide.
  -> [('poulet', 0.48), ('tajine', 0.45), ('délicieux', 0.45)]


In [42]:
# ✅ Vérification (ne pas modifier)
verifier("k mots, du plus grand score au plus petit", lambda: [s for _, s in mots_caracteristiques(0, 4)] == sorted([s for _, s in mots_caracteristiques(0, 4)], reverse=True))

✅ k mots, du plus grand score au plus petit


### 📝 Interprétation 2
**(a)** « service » a un IDF de **2,18** (présent dans 30 avis sur 100) contre **4,52** pour « kefta » (2 avis seulement). L'IDF mesure le pouvoir discriminant d'un mot : un mot présent partout n'aide pas à distinguer les documents (« service » apparaît dans tous les types d'avis, positifs comme négatifs), alors qu'un mot rare comme « kefta » identifie immédiatement les rares avis qui en parlent, d'où un poids plus de deux fois supérieur.
« service » ne vaut pas 0 ici, contrairement au cours, pour deux raisons : (1) il n'est pas dans tous les documents (30/100), alors qu'il était dans les 3 documents du cours, donc log₁₀(3/3) = 0 ; (2) scikit-learn utilise la formule **lissée** idf = ln((1+N)/(1+df)) + 1, qui est toujours ≥ 1 : aucun mot n'est jamais complètement ignoré (et évite la division par 0 pour un mot inconnu). Notons aussi que scikit-learn utilise le logarithme népérien et normalise chaque ligne (norme 1), ce qui change les valeurs mais pas leur ordre.

**(b)** Résumé partiellement bon seulement.
- Avis 0 (*tajine délicieux… service très rapide*) → `tajine, délicieux, recommande` : bon résumé, mais « service rapide » est relégué car « service » est un mot fréquent.
- Avis 63 → `poulet, tajine, délicieux` : très bon (le plat et le jugement).
- Avis 3 (*couscous n'était pas bon du tout, trop salé*) → `salé, tout, couscous` : on perd « pas bon » (la négation a un IDF faible) et « tout » est du bruit ; le résumé ne dit plus que l'avis est négatif.
- Avis 40 → `yeux, fermés, excellent` : les mots rares de l'expression « les yeux fermés » dominent, alors que l'information utile est « excellent rapport qualité prix ».
Conclusion : TF-IDF met en avant les mots **rares**, pas forcément les mots **importants** ; il est bon pour identifier le sujet (plat, produit) mais faible pour le sentiment et les expressions.

## Partie 3 — Similarité cosinus
### Exercice 3.1 ★ Le cosinus à la main

In [43]:
def cosinus(u, v):
    nu, nv = np.linalg.norm(u), np.linalg.norm(v)
    return 0 if nu == 0 or nv == 0 else float(u @ v / (nu * nv))
print(round(cosinus(np.array([1, 1, 0]), np.array([1, 0, 1])), 3))
u, v = X_bow[0].toarray()[0], X_bow[27].toarray()[0]
print(round(cosinus(u, v), 3), "| scikit-learn :", round(cosine_similarity(X_bow[0], X_bow[27])[0, 0], 3))

0.5
0.73 | scikit-learn : 0.73


In [44]:
# ✅ Vérification (ne pas modifier)
verifier("cos((1,1,0), (1,0,1)) = 0,5", lambda: abs(cosinus(np.array([1, 1, 0]), np.array([1, 0, 1])) - 0.5) < 1e-12)
verifier("vecteur nul", lambda: cosinus(np.zeros(3), np.ones(3)) == 0)
verifier("comme scikit-learn", lambda: abs(cosinus(u, v) - cosine_similarity(X_bow[0], X_bow[27])[0, 0]) < 1e-9)

✅ cos((1,1,0), (1,0,1)) = 0,5
✅ vecteur nul
✅ comme scikit-learn


### Exercice 3.2 ★★ Les deux avis les plus proches

In [45]:
S = cosine_similarity(X_tfidf)
S2 = S.copy()
np.fill_diagonal(S2, 0)
i, j = np.unravel_index(np.argmax(S2), S2.shape)
print(f"cosinus = {S2[i, j]:.2f}"); print(i, textes[i]); print(j, textes[j])

cosinus = 0.94
27 Le tajine au poulet était délicieux, service rapide.
63 Le tajine au poulet était délicieux et le service très rapide.


In [46]:
# ✅ Vérification (ne pas modifier)
verifier("matrice 100 × 100", lambda: S.shape == (100, 100) and np.allclose(np.diag(S), 1))
verifier("le maximum hors diagonale", lambda: i != j and np.isclose(S2[i, j], S2.max()))

✅ matrice 100 × 100
✅ le maximum hors diagonale


### Exercice 3.3 ★★ Les voisins d'un avis

In [47]:
def voisins(i, k=3):
    scores = S[i].copy()
    scores[i] = -1  # on exclut l'avis lui-même
    ordre = np.argsort(scores)[::-1][:k]
    return [(round(float(scores[j]), 2), int(j)) for j in ordre]
print(textes[1])
for s, j in voisins(1):
    print(" ", s, textes[j])

Service TRÈS lent, le plat est arrivé froid. Je ne recommande pas.
  0.81 Service lent, le plat était froid. Je ne recommande pas.
  0.55 Le plat est arrivé froid et le service était lent, quelle déception.
  0.49 Le couscous était froid et le service lent.


In [48]:
# ✅ Vérification (ne pas modifier)
verifier("k voisins, sans l'avis lui-même", lambda: len(voisins(1, 5)) == 5 and 1 not in [j for _, j in voisins(1, 5)])

✅ k voisins, sans l'avis lui-même


In [49]:
# Pour l'interprétation 3 : on regarde aussi les voisins de rang 4 à 6
for s, j in voisins(1, 6)[3:]:
    print(" ", s, f"[{etiquettes[j]}]", textes[j])

  0.4 [neg] Attente interminable, service lent et serveur désagréable. Je ne recommande pas.
  0.33 [pos] Très bon accueil, service rapide, je recommande.
  0.31 [pos] Le tajine était délicieux et le service très rapide. Je recommande !


### 📝 Interprétation 3
**Les deux avis les plus proches** (cosinus = 0,94) sont l'avis 27 *« Le tajine au poulet était délicieux, service rapide. »* et l'avis 63 *« Le tajine au poulet était délicieux et le service très rapide. »*. C'est pertinent : ils disent presque la même chose, et partagent exactement les mots `tajine, poulet, délicieux, service, rapide` ; seul « très » diffère.

**Voisins de l'avis 1** (*Service TRÈS lent, le plat est arrivé froid. Je ne recommande pas.*) : l'avis 29 (0,81, quasi identique), l'avis 80 (0,55) et l'avis 32 (0,49) sont tous négatifs et parlent de service lent et de plat froid : très pertinents. La similarité repose sur les mots `service, lent, froid, ne, recommande, pas`.
**Voisin moins pertinent** : à partir du rang 5, on trouve l'avis 31 *« Très bon accueil, service rapide, je recommande. »* (≈ 0,33, **positif**) puis l'avis 0 (≈ 0,31, positif aussi). Ils ne partagent que « service » et « recommande », alors que le sens est opposé : dans l'avis 1, « recommande » est nié par « ne … pas ». Le sac de mots ignore l'ordre et la portée de la négation, donc il rapproche des avis de sentiments contraires dès qu'ils ont des mots en commun.

## Partie 4 — Le mini moteur de recherche (livrable)

In [50]:
def search(query, k=5):
    q = vec_tfidf.transform([query])
    scores = cosine_similarity(q, X_tfidf)[0]
    ordre = [j for j in np.argsort(scores)[::-1][:k] if scores[j] > 0]
    return pd.DataFrame({"score": [round(float(scores[j]), 3) for j in ordre],
                         "avis": [textes[j] for j in ordre],
                         "étiquette": [etiquettes[j] for j in ordre]})
search("service lent")

,score,avis,étiquette
0,0.606,Le couscous était froid et le service lent.,neg
1,0.606,Le service était lent mais les plats étaient bons.,pos
2,0.530,"Service lent, le plat était froid. Je ne recommande pas.",neg
3,0.458,"Rapport qualité prix correct, mais le service est lent.",neg
4,0.430,"Service TRÈS lent, le plat est arrivé froid. Je ne recommande pas.",neg


In [51]:
# ✅ Vérification (ne pas modifier)
_r = search("service lent")
verifier("colonnes", lambda: list(_r.columns) == ["score", "avis", "étiquette"])
verifier("scores décroissants", lambda: list(_r["score"]) == sorted(_r["score"], reverse=True) and len(_r) == 5)
verifier("le premier avis contient 'lent'", lambda: "lent" in _r.loc[0, "avis"].lower())
verifier("requête sans mot connu : tableau vide", lambda: len(search("zzz")) == 0)

✅ colonnes
✅ scores décroissants
✅ le premier avis contient 'lent'
✅ requête sans mot connu : tableau vide


### Les 6 requêtes de test

In [52]:
REQUETES = ["tajine au poulet délicieux", "livraison en retard", "pas cher", "accueil chaleureux à Rabat",
            "restaurent magnifque", "service vite"]
for r in REQUETES:
    res = search(r, 3)
    print(f"\n🔎 {r} ({len(res)} résultat(s))")
    for _, ligne in res.iterrows():
        print(f"  {ligne['score']:.2f} [{ligne['étiquette']}] {ligne['avis']}")


🔎 tajine au poulet délicieux (3 résultat(s))
  0.85 [pos] Le tajine au poulet était délicieux, service rapide.
  0.80 [pos] Le tajine au poulet était délicieux et le service très rapide.
  0.60 [pos] Personnel souriant, tajine au poulet délicieux, très bon moment.

🔎 livraison en retard (3 résultat(s))
  0.61 [neg] La commande était incomplète et la livraison en retard d'une heure.
  0.54 [neg] Livraison en retard, le livreur était impoli et le repas froid.
  0.52 [neg] Livraison en retard de 45 minutes, la commande était incomplète.

🔎 pas cher (3 résultat(s))
  0.59 [pos] Pas cher, très bon, je recommande ce restaurant à Rabat.
  0.58 [pos] Le tajine au poulet est excellent et pas cher.
  0.56 [neg] Pas cher mais pas bon : la viande était dure et trop salée.

🔎 accueil chaleureux à Rabat (3 résultat(s))
  0.42 [pos] Accueil chaleureux et service rapide, le tajine au poulet est un régal.
  0.37 [pos] Ambiance familiale, accueil chaleureux et tajine au poulet délicieux.
  0.36 [pos] P

### Précision@3 (jugement manuel de la pertinence)
Un résultat est jugé pertinent s'il répond bien à l'intention de la requête.

In [53]:
# 1 = pertinent, 0 = non pertinent, pour chacun des (au plus) 3 résultats affichés ci-dessus
jugements = {
    "tajine au poulet délicieux": [1, 1, 1],
    "livraison en retard":        [1, 1, 1],
    "pas cher":                   [1, 1, 0],   # le 3e dit « pas cher mais pas bon » : mot présent, sens négatif
    "accueil chaleureux à Rabat": [1, 1, 0],   # le 3e ne parle que de Rabat
    "restaurent magnifque":       [1],         # un seul résultat (le seul avis avec la même faute)
    "service vite":               [0, 0, 1],   # « vite » inconnu ; seuls les avis « rapide » conviennent
}
for r, j in jugements.items():
    print(f"P@{len(j)} = {sum(j)}/{len(j)} = {sum(j)/len(j):.2f}   ← {r}")
print(f"\nP@3 moyenne (sur les 6 requêtes) = {np.mean([sum(j)/len(j) for j in jugements.values()]):.2f}")

P@3 = 3/3 = 1.00   ← tajine au poulet délicieux
P@3 = 3/3 = 1.00   ← livraison en retard
P@3 = 2/3 = 0.67   ← pas cher
P@3 = 2/3 = 0.67   ← accueil chaleureux à Rabat
P@1 = 1/1 = 1.00   ← restaurent magnifque
P@3 = 1/3 = 0.33   ← service vite

P@3 moyenne (sur les 6 requêtes) = 0.78


### 📝 Analyse du moteur (livrable)
**Réussites.**
- « tajine au poulet délicieux » (P@3 = 3/3) et « livraison en retard » (3/3) : les mots de la requête sont rares et discriminants, les meilleurs résultats contiennent les trois (ou les deux) mots et le classement est cohérent (scores 0,85 → 0,60 pour le tajine).
- « accueil chaleureux à Rabat » : les deux premiers résultats contiennent « accueil chaleureux » ; « à » est un mot vide ignoré.
- « zzz » (mot inconnu) renvoie un tableau vide, plutôt que des résultats au hasard.

**Échecs.**
1. **Synonymes (« service vite »)** : « vite » n'est pas dans le vocabulaire (le corpus dit « rapide »). Le moteur ne retient que « service » et renvoie des avis qui parlent de service **lent** (P@3 = 1/3) : il y a un décalage de vocabulaire (*vocabulary mismatch*), le sac de mots ne connaît pas le sens.
2. **Fautes d'orthographe (« restaurent magnifque »)** : un seul résultat, l'avis qui contient exactement la même faute. Les avis qui écrivent correctement « magnifique » (cadre magnifique…) sont introuvables, car « magnifique » et « magnifque » sont deux colonnes différentes.
3. **Négation et expressions (« pas cher »)** : le moteur compte « pas » et « cher » séparément. Le 3e résultat, *« Pas cher mais pas bon… »*, contient les deux mots mais exprime un avis négatif ; de plus un avis contenant « pas » et « cher » dans un autre ordre/contexte (« pas bon… trop cher ») serait aussi rapproché.
4. **Mots hors sujet (« … à Rabat »)** : le 3e résultat n'est là que grâce à « Rabat », sans parler d'accueil chaleureux.

**Améliorations possibles** : bigrammes (ci-dessous, « pas cher » devient un seul terme), lemmatisation/racinisation (*recommande/recommandé*), n-grammes de caractères (`analyzer="char_wb"`) pour tolérer les fautes, correction orthographique de la requête, expansion de requête par synonymes, ou plongements de mots (*embeddings*) pour capter le sens.

## 🚀 Pour aller plus loin — A. Bigrammes

In [54]:
def analyseur_bi(texte):
    m = analyseur(texte)
    return m + [" ".join(m[i:i + 2]) for i in range(len(m) - 1)]
vec_bi = TfidfVectorizer(analyzer=analyseur_bi)
X_bi = vec_bi.fit_transform(textes)
s = cosine_similarity(vec_bi.transform(["pas cher"]), X_bi)[0]
for j in np.argsort(s)[::-1][:3]:
    print(round(float(s[j]), 2), etiquettes[j], textes[j])

0.5 pos Le tajine au poulet est excellent et pas cher.
0.47 pos Pas cher, très bon, je recommande ce restaurant à Rabat.
0.46 pos Très bon restaurant, les desserts sont délicieux et pas cher.


**Constat :** avec les bigrammes, les trois premiers résultats pour « pas cher » contiennent tous l'expression « pas cher » **consécutive** et sont tous positifs. L'avis négatif « Pas cher mais pas bon… » sort du top 3 : le bigramme `pas cher` capte un peu du contexte que le sac de mots simple perd. Le coût : un vocabulaire plus grand (donc une matrice plus creuse).

## ✔️ Avant de rendre
- [x] Identification à adapter (classe, groupe, site) ; noyau → *Redémarrer et tout exécuter*
- [x] `search(query)` testée sur les 6 requêtes ; P@3 calculées
- [x] Interprétations rédigées, dont l'analyse du moteur